# Face Occlusion — Kaggle Training Notebook

## 1. Install & Clone

In [ ]:
import os

!pip install timm wandb tqdm -q

!git clone https://github.com/LeHoang510/Face-Occlusion-Prediction.git
os.chdir("/kaggle/working/Face-Occlusion-Prediction")
print("Working dir:", os.getcwd())

## 2. Prepare Dataset

In [ ]:
import os

KAGGLE_INPUT = "/kaggle/input/face-occlusion-data"

# Images
os.makedirs("dataset/crops", exist_ok=True)
!unzip -q {KAGGLE_INPUT}/crops.zip -d dataset/crops/

# CSVs
os.makedirs("dataset/DataChallenge2026/occlusion_datasets", exist_ok=True)
!cp {KAGGLE_INPUT}/train.csv dataset/DataChallenge2026/occlusion_datasets/
!cp {KAGGLE_INPUT}/test_students.csv dataset/DataChallenge2026/occlusion_datasets/

# Verify
print("Images:", sum(len(f) for _, _, f in os.walk("dataset/crops")), "files")
print("Train CSV:", len(open("dataset/DataChallenge2026/occlusion_datasets/train.csv").readlines()) - 1, "samples")

## 3. WandB Login

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("WandB API key loaded.")

## 4. Configure & Train

Edit `CONFIG` below to pick the model you want to train.

In [ ]:
import yaml

# ── Pick your config ──────────────────────────────────────────────────
CONFIG = "src/data_challenge/configs/dinov2_small.yaml"
# CONFIG = "src/data_challenge/configs/dinov2_base.yaml"
# CONFIG = "src/data_challenge/configs/swin_s.yaml"
# ─────────────────────────────────────────────────────────────────────

# Redirect output to /kaggle/working/ so it's preserved after the session
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

cfg["output"]["dir"] = "/kaggle/working/outputs"

with open(CONFIG, "w") as f:
    yaml.dump(cfg, f)

print(f"Config: {CONFIG}")
print(f"Backbone: {cfg['model']['backbone']}")
print(f"Output dir: {cfg['output']['dir']}")

In [ ]:
!python -m data_challenge.train --config {CONFIG}

## 5. Save Checkpoint to WandB Artifact

In [ ]:
import glob, json, wandb

summary_path = glob.glob("/kaggle/working/outputs/*/run_summary.json")[0]
best_ckpt    = glob.glob("/kaggle/working/outputs/*/best_model.pt")[0]
config_path  = glob.glob("/kaggle/working/outputs/*/config.yaml")[0]

with open(summary_path) as f:
    summary = json.load(f)

run = wandb.init(
    project="data-challenge-occlusion",
    id=summary["wandb_run_id"],
    resume="must",
)
artifact = wandb.Artifact(summary["run_name"], type="model", metadata=summary)
artifact.add_file(best_ckpt,   name="best_model.pt")
artifact.add_file(summary_path, name="run_summary.json")
artifact.add_file(config_path,  name="config.yaml")
run.log_artifact(artifact)
run.finish()

print(f"Artifact '{summary['run_name']}' saved to wandb.")
print(f"Best score: {summary['best_val_score']:.6f} at epoch {summary['best_epoch']}")

## 6. Pull checkpoint back to local machine

Run this on your **local machine** after the Kaggle run finishes:

```bash
.venv/bin/python -c "
import wandb
api = wandb.Api()
artifact = api.artifact('lehoang510/data-challenge-occlusion/dinov2_small:latest')
artifact.download('outputs/dinov2_small_kaggle/')
print('Done.')
"
```